In [47]:
#Importing necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
import numpy as np
import os
import h5py

current_dir = os.getcwd()

#Loading GT and SUB Data
try:
    #Output/GT_h5/groundtruth.h5
    folder_dir = "output/GT_h5"
    h5_file_path = os.path.join(current_dir, "output", "GT_h5", "groundtruth.h5")
    gt_data  = pd.read_hdf(h5_file_path, "/data")
    print("Voltage Data Loading done!")
    gt_spikes = pd.read_hdf(h5_file_path,"/spikes_raw")
    print("Spikes Data Loading done!")

    sub_data = gt_data
    sub_spikes = gt_spikes

    #Common to both GT and SUB
    cfg  = pd.read_hdf(h5_file_path, "/network_config")
    print("Network configuration Loading done!")
    tmap = pd.read_hdf(h5_file_path, "/trial_map")
    print("Trial Map Loading done!")

    import json
    with open("output/GT/network_config.json", "r") as f:
        net_config = json.load(f)
    truth_table = net_config["truth_table"]

except Exception as e:
    print(f"Data laoding failed as {e}")


Voltage Data Loading done!
Spikes Data Loading done!
Network configuration Loading done!
Trial Map Loading done!


In [48]:
try:
    # verify GT and SUB have same number of trials
    assert len(gt_data["trial_id"].unique()) == len(sub_data["trial_id"].unique())

    # verify pattern distribution is identical
    assert (gt_data.groupby("case").size() == sub_data.groupby("case").size()).all()

except Exception as e:
    print(e)

In [49]:
#Helper Functions
def get_neurons(cfg, role=None, input_channel=None):
    #     role="output"                 - ["E"]
    #     role="input"                  - ["PyrIn_A","PyrIn_B1","PyrIn_B2"]
    #     role=["input","interneuron",
    #           "intermediate","output"]- all 8 active neurons
    #     role="all"                    - all 35 labels
    #     input_channel=2               - ["PyrIn_B1","PyrIn_B2"]
    result = cfg
    if role!= None and role!= "all":
        if isinstance(role, list):
            result = result[result["role"].isin(role)]
        else:
            result = result[result["role"] == role]

    if input_channel!= None:
        result = result[result["input_channel"] == input_channel]

    return result["label"].tolist()
        
def get_spiking_neurons(cfg, data, label = None):
    #     checks _spike cols in data at runtime
    #     returns labels where spike col sum > 0
    spiking = []

    if label == None:
        for _ in cfg["label"]:
            if data[f"{_}_spike"].sum()>0:
                spiking.append(_)
    else:
        if data[label].sum() > 0:
            spiking.append(label)
       
    return spiking
        
def get_spike_cols(cfg, data, role =None):
   #     calls get_spiking_neurons()
   #     returns ["{label}_spike", ...] for spiking neurons
   columns = []
   
   if role == None:
        for label in get_spiking_neurons(cfg,data):
            columns.append(f"{label}_spike")
    
   else:
        for label in get_neurons(cfg, role=role):
            columns.append(f"{label}_spike")
    
   return columns

def get_vm_cols(cfg, scope="all"):
    #     scope="all"    - all 35 _vm column names
    #     scope="active" - 8 active neurons _vm column names
    roles = []
    if(scope == "active"):
        for _,row in cfg.iterrows():
            if row["role"] != "extended_network":
                roles.append(f"{row['label']}_vm")
    else:
        for _,row in cfg.iterrows():
            roles.append(f"{row['label']}_vm")
    return roles

def get_trial(data, trial_id):
    return data[data["trial_id"] == trial_id]

def get_trials_by_pattern(data, pattern):
    #     returns list of DataFrames where case == pattern
    #     10 DataFrames per pattern (one per rep)
    result = []
    unique = data["rep"].unique()
    for u in unique:
        filtered = data[(data["case"] == pattern) & (data["rep"] == u)]
        result.append(filtered)

    return result

def load_metadata(h5_path):
    #     reads /metadata attrs via h5py
    #     returns plain dict:
    #       {"fs_hz": 1000.0, "t_total_ms": 4000.0, "n_trials": 40,
    #        "trial_len_ms": 100.0, "n_neurons_total": 35, "n_neurons_spiking": 8}
    with h5py.File(h5_path,"r") as f:
        metadata = dict(f["/metadata"].attrs)
    
    return metadata

In [ ]:
#Behavioural Metrics
#This metric needs the output neuron
role_behavioural = "output"
patterns = ["00","01","10","11"]


print(truth_table)
#print("The neuron name is :",get_neurons(cfg, role=role_behavioural)) #Veryifing whether we are getting the correct neuron or not
#print("The neuron label is:",get_spike_cols(cfg, gt_data, role=role_behavioural)) #Veryifing whether we are getting the correct neuron label or not
label = get_neurons(cfg, role=role_behavioural)
label = get_spike_cols(cfg, gt_data, role=role_behavioural)

meta = load_metadata(h5_file_path)
trial_len = meta["trial_len_ms"]

def compute_confusion_matrix(data,pattern):
        trials = get_trials_by_pattern(data,pattern)
        count_fired = 0
        count_not_fired = 0
        for t in trials:
            #print(t["t_in_trial"])
            window = t[t["t_in_trial"] <= trial_len]
            fired = window[label].sum() > 0

        
def all_patterns(data,patterns):
    for p in patterns:
        compute_confusion_matrix(data,p)

{'XOR_00': {'input_A': 0, 'input_B': 0, 'expected_output': 0}, 'XOR_01': {'input_A': 0, 'input_B': 1, 'expected_output': 1}, 'XOR_10': {'input_A': 1, 'input_B': 0, 'expected_output': 1}, 'XOR_11': {'input_A': 1, 'input_B': 1, 'expected_output': 0}}
